# Fine-tune Llama-3.1-8B with Unsloth on Google Colab

Notebook này giúp bạn train Llama-3.1-8B với Unsloth + LoRA trên Google Colab, chỉ cần upload 2 folder: `data_pipeline` và `llm_finetuning_serving`. Checkpoint sẽ được lưu vào Google Drive sau mỗi epoch để tránh mất mát khi Colab bị ngắt.

## 1. Mount Google Drive and Setup Environment
Mount Google Drive để lưu checkpoint/model và thiết lập biến môi trường cơ bản.

In [ ]:
# 1. Mount Google Drive (phải mount trước khi cài requirements!)
from google.colab import drive
drive.mount('/content/drive')

# Đặt lại các biến đường dẫn code/data/output
import os
DRIVE_CODE_DIR = '/content/drive/MyDrive/llama_code'
DRIVE_DATA_PIPELINE = f'{DRIVE_CODE_DIR}/data_pipeline'
DRIVE_FINETUNE = f'{DRIVE_CODE_DIR}/llm_finetuning_serving/finetune'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/llama_finetune_outputs'
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.environ['OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
print(f"Đã mount Drive và set biến đường dẫn: {DRIVE_CODE_DIR}")

## 2. Install Required Packages
Cài đặt các thư viện cần thiết cho Unsloth, Transformers, TRL, Datasets, WandB, v.v.

In [ ]:
# 2. Thêm code vào sys.path trước khi pip install requirements
import sys
sys.path.append('/content/drive/MyDrive/llama_code/llm_finetuning_serving/finetune')
sys.path.append('/content/drive/MyDrive/llama_code/data_pipeline')

# 3. Cài đặt các thư viện cần thiết
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth transformers trl datasets wandb
# Nếu có requirements.txt trong 2 folder, cài luôn:
!pip install -r /content/drive/MyDrive/llama_code/llm_finetuning_serving/finetune/requirements.txt || true
!pip install -r /content/drive/MyDrive/llama_code/data_pipeline/requirements.txt || true

## 3. Clone or Upload Code Folders
Bạn có thể upload trực tiếp 2 folder `data_pipeline` và `llm_finetuning_serving` lên Colab hoặc dùng lệnh dưới nếu đã đẩy lên Google Drive hoặc Git.

In [ ]:
# Code và data đã ở Google Drive, chạy trực tiếp trên Drive cho đồng bộ
import sys, os
DRIVE_CODE_DIR = '/content/drive/MyDrive/llama_code'
DRIVE_DATA_PIPELINE = f'{DRIVE_CODE_DIR}/data_pipeline'
DRIVE_FINETUNE = f'{DRIVE_CODE_DIR}/llm_finetuning_serving/finetune'

sys.path.append(DRIVE_FINETUNE)
sys.path.append(DRIVE_DATA_PIPELINE)

print(f"Đã thêm code vào sys.path: {DRIVE_FINETUNE}, {DRIVE_DATA_PIPELINE}")

## 4. Set Up Paths and Imports
Thêm 2 folder vào sys.path và import các module cần thiết cho training.

In [ ]:
import sys
import os
from pathlib import Path
from datetime import datetime
import torch
import json
import logging
import wandb
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import Dataset

# Đảm bảo sys.path đã thêm đúng folder code trên Drive
DRIVE_CODE_DIR = '/content/drive/MyDrive/llama_code'
DRIVE_DATA_PIPELINE = f'{DRIVE_CODE_DIR}/data_pipeline'
DRIVE_FINETUNE = f'{DRIVE_CODE_DIR}/llm_finetuning_serving/finetune'
sys.path.append(DRIVE_FINETUNE)
sys.path.append(DRIVE_DATA_PIPELINE)
print(f"sys.path đã thêm: {DRIVE_FINETUNE}, {DRIVE_DATA_PIPELINE}")

## 5. Configure Training Parameters for Colab
Thiết lập cấu hình training phù hợp với GPU Colab (batch size nhỏ, output lưu trên Drive).

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class FineTuneConfig:
    model_name: str = "unsloth/Llama-3.1-8B-Instruct"
    max_seq_length: int = 4096
    dtype: Optional[torch.dtype] = None
    load_in_4bit: bool = True
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    bias: str = "none"
    use_gradient_checkpointing: str = "unsloth"
    random_state: int = 3407
    use_rslora: bool = True
    per_device_train_batch_size: int = 2
    per_device_eval_batch_size: int = 2
    gradient_accumulation_steps: int = 8
    warmup_steps: int = 50
    num_train_epochs: int = 3
    max_steps: int = -1
    learning_rate: float = 2e-4
    weight_decay: float = 0.01
    lr_scheduler_type: str = "cosine"
    seed: int = 3407
    optim: str = "adamw_torch"
    fp16: bool = True
    bf16: bool = False
    dataloader_num_workers: int = 2
    dataloader_pin_memory: bool = True
    logging_steps: int = 10
    save_strategy: str = "epoch"
    evaluation_strategy: str = "epoch"
    save_total_limit: int = 2
    load_best_model_at_end: bool = True
    metric_for_best_model: str = "eval_loss"
    greater_is_better: bool = False
    dataset_text_field: str = "text"
    packing: bool = False
    # Output lưu trên Drive
    output_dir: str = '/content/drive/MyDrive/llama_finetune_outputs'
    run_name: str = field(default_factory=lambda: f"colab-llama-{datetime.now().strftime('%Y%m%d-%H%M%S')}")

config = FineTuneConfig()

## 6. Load and Prepare Datasets
Load dữ liệu train/val/test từ data_pipeline hoặc từ file splits.

In [ ]:
from datasets import Dataset
import json
from pathlib import Path

# Đọc data trực tiếp từ Drive
DRIVE_CODE_DIR = '/content/drive/MyDrive/llama_code'
DRIVE_DATA_PIPELINE = f'{DRIVE_CODE_DIR}/data_pipeline'
splits_dir = Path(f'{DRIVE_DATA_PIPELINE}/data/finetune_data')

def load_jsonl_dataset(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return Dataset.from_list(data)

train_path = splits_dir / 'train.jsonl'
val_path = splits_dir / 'valid.jsonl'
test_path = splits_dir / 'test.jsonl'

datasets = {}
if train_path.exists():
    datasets['train'] = load_jsonl_dataset(train_path)
if val_path.exists():
    datasets['val'] = load_jsonl_dataset(val_path)
if test_path.exists():
    datasets['test'] = load_jsonl_dataset(test_path)

print(f"Train: {len(datasets.get('train', []))}")
print(f"Val: {len(datasets.get('val', []))}")
print(f"Test: {len(datasets.get('test', []))}")

In [ ]:
# Nếu bạn đã có train.jsonl và muốn chia thành valid.jsonl và test.jsonl
import json, random, os
from pathlib import Path

splits_dir = Path('/content/drive/MyDrive/llama_code/data_pipeline/data/finetune_data')
train_path = splits_dir / 'train.jsonl'
val_path = splits_dir / 'valid.jsonl'
test_path = splits_dir / 'test.jsonl'

if train_path.exists():
    with open(train_path, 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]
    random.shuffle(data)
    n = len(data)
    val_size = int(0.05 * n)
    test_size = int(0.05 * n)
    train_size = n - val_size - test_size
    train_data = data[:train_size]
    val_data = data[train_size:train_size+val_size]
    test_data = data[train_size+val_size:]
    with open(train_path, 'w', encoding='utf-8') as f:
        for ex in train_data:
            f.write(json.dumps(ex, ensure_ascii=False)+'\n')
    with open(val_path, 'w', encoding='utf-8') as f:
        for ex in val_data:
            f.write(json.dumps(ex, ensure_ascii=False)+'\n')
    with open(test_path, 'w', encoding='utf-8') as f:
        for ex in test_data:
            f.write(json.dumps(ex, ensure_ascii=False)+'\n')
    print(f"Đã chia: train={len(train_data)}, val={len(val_data)}, test={len(test_data)}")
else:
    print("Không tìm thấy train.jsonl để chia!")

In [ ]:
# Tải thử data mẫu từ HuggingFace Datasets (ví dụ: alpaca, hoặc instruct data)
from datasets import load_dataset

# Ví dụ: tải dataset alpaca (có thể đổi sang dataset khác nếu muốn)
hf_dataset = load_dataset('tatsu-lab/alpaca')

# Chuyển đổi sang định dạng train/val/test jsonl nếu cần
import json
os.makedirs('/content/drive/MyDrive/llama_code/data_pipeline/data/finetune_data', exist_ok=True)
with open('/content/drive/MyDrive/llama_code/data_pipeline/data/finetune_data/train.jsonl', 'w', encoding='utf-8') as f:
    for ex in hf_dataset['train']:
        f.write(json.dumps(ex, ensure_ascii=False) + '\n')
print('Đã tải và lưu data mẫu vào Drive!')

## 7. Load Model with Unsloth
Load model Llama-3.1-8B với Unsloth và cấu hình LoRA phù hợp Colab.

In [ ]:
# Load model và tokenizer với Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config.model_name,
    max_seq_length=config.max_seq_length,
    dtype=config.dtype,
    load_in_4bit=config.load_in_4bit,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

# Áp dụng LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=config.lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias=config.bias,
    use_gradient_checkpointing=config.use_gradient_checkpointing,
    random_state=config.random_state,
    use_rslora=config.use_rslora,
    max_seq_length=config.max_seq_length,
)
print("Model & LoRA ready!")

## 8. Apply Chat Template to Datasets
Format lại dữ liệu theo template hội thoại cho Llama.

In [ ]:
# Hàm format hội thoại cho Llama

def formatting_prompts_func(examples):
    texts = []
    for i in range(len(examples['instruction'])):
        user_input = f"{examples['instruction'][i]} {examples['input'][i]}"
        text = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a helpful assistant that answers questions about Vietnamese law.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{examples['output'][i]}<|eot_id|>"""
        texts.append(text)
    return {"text": texts}

formatted_datasets = {}
for split_name, dataset in datasets.items():
    formatted_dataset = dataset.map(
        formatting_prompts_func,
        batched=True,
        remove_columns=dataset.column_names
    )
    formatted_datasets[split_name] = formatted_dataset
    print(f"{split_name}: {len(formatted_dataset)} examples")
    print("Ví dụ:", formatted_dataset[0]['text'][:300])

## 9. Setup Trainer with Save-Per-Epoch
Cấu hình Trainer để checkpoint tự động lưu vào Google Drive sau mỗi epoch.

In [ ]:
import os
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

# 1. Cấu hình để lưu định kỳ và tiết kiệm VRAM tối đa
training_args = TrainingArguments(
    output_dir=config.output_dir,
    run_name=config.run_name,
    
    # --- CHỐNG SẬP & LƯU TRỮ ---
    save_strategy="epoch",           # Lưu file model sau mỗi lần chạy hết 1 vòng dữ liệu
    save_total_limit=2,              # Chỉ giữ 2 bản gần nhất để không bị đầy dung lượng Drive/Colab
    
    # --- TẮT EVAL ĐỂ TRÁNH LỖI VALUEERROR ---
    eval_strategy="no",              # Không cần tập validation, tránh lỗi như lúc nãy
    load_best_model_at_end=False,    # Tắt cái này vì không có eval để so sánh
    
    # --- ÉP VRAM XUỐNG THẤP NHẤT (CHỐNG OOM) ---
    per_device_train_batch_size=1,    # Batch size nhỏ nhất
    gradient_accumulation_steps=4,    # Gom 4 lần chạy thành 1 lần cập nhật (tương đương batch size 4)
    optim="paged_adamw_8bit",         # Optimizer tiết kiệm bộ nhớ cho Unsloth
    gradient_checkpointing=True,      # Tiết kiệm VRAM khi lan truyền ngược
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # --- CÁC THAM SỐ KHÁC ---
    learning_rate=2e-4,
    num_train_epochs=3,               # Bạn có thể để 3 epoch như dự định
    logging_steps=1,
    seed=config.seed,
    report_to=["wandb"] if os.getenv("WANDB_API_KEY") else [],
)

# 2. Khởi tạo Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_datasets['train'],
    eval_dataset=None,               # Không cần truyền tập test
    dataset_text_field=config.dataset_text_field,
    max_seq_length=512,               # Giới hạn độ dài để an toàn cho GPU 15GB
    args=training_args,
    packing=False,
)

print("Đã sẵn sàng train và lưu sau mỗi Epoch!")

# 3. Bắt đầu huấn luyện
trainer.train()

In [ ]:
# Đặt WANDB_API_KEY nếu bạn muốn dùng WandB tracking
import os
os.environ["WANDB_API_KEY"] = "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"  # Thay bằng key thật của bạn hoặc dùng os.environ["WANDB_API_KEY"] = input("Nhập WANDB_API_KEY: ")

## 10. Train Model with Checkpointing
Chạy training, checkpoint sẽ tự động lưu vào Google Drive sau mỗi epoch. Nếu Colab bị ngắt, có thể resume từ checkpoint.

In [ ]:
# Nếu muốn resume từ checkpoint, chỉ cần truyền checkpoint path vào trainer.train()
# Ví dụ: trainer.train(resume_from_checkpoint='/content/drive/MyDrive/llama_finetune_outputs/checkpoint-xxx')

train_result = trainer.train()
print("Training done!")

# Lưu lại kết quả training
with open(os.path.join(config.output_dir, 'train_results.json'), 'w') as f:
    json.dump(train_result.metrics, f, indent=2)

## 11. Evaluate Model
Đánh giá model trên validation/test set và lưu kết quả vào Drive.

In [ ]:
# Đánh giá trên validation set
if formatted_datasets.get('val') is not None:
    val_metrics = trainer.evaluate()
    print("Validation:", val_metrics)
    with open(os.path.join(config.output_dir, 'val_metrics.json'), 'w') as f:
        json.dump(val_metrics, f, indent=2)

# Đánh giá trên test set nếu có
test_metrics = None
if formatted_datasets.get('test') is not None:
    test_metrics = trainer.evaluate(formatted_datasets['test'])
    print("Test:", test_metrics)
    with open(os.path.join(config.output_dir, 'test_metrics.json'), 'w') as f:
        json.dump(test_metrics, f, indent=2)

## 12. Save and Download Final Model
Lưu model/tokenizer cuối cùng vào Google Drive và hướng dẫn tải về.

In [ ]:
# Lưu model/tokenizer cuối cùng
final_model_dir = os.path.join(config.output_dir, 'final_model')
os.makedirs(final_model_dir, exist_ok=True)
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"Model & tokenizer saved to {final_model_dir}")

# Hướng dẫn tải về
from google.colab import files
print(f"Bạn có thể tải model/tokenizer về bằng lệnh sau hoặc tải thủ công từ Google Drive:")
print(f"!zip -r final_model.zip {final_model_dir}")
# files.download('final_model.zip')  # Bỏ comment nếu muốn tải trực tiếp